# Dynamic Memory

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Allocate memory whose size is known only while the program runs, with malloc and calloc, check for NULL, and free it.
- Grow a buffer with realloc, doubling its capacity, and explain why the old pointer must not be used afterwards.
- Build arrays of rows with different lengths, or one block indexed by row and column.

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check_int and check_str print ✓ or ✗; the exercises below use them
void check_int(const char *what, long actual, long expected) {
    if (actual == expected) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: expected %ld, got %ld\n", what, expected, actual);
    }
}

void check_str(const char *what, const char *actual, const char *expected) {
    if (actual == NULL) {
        actual = "(NULL)";
    }
    if (strcmp(actual, expected) == 0) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: expected \"%s\", got \"%s\"\n", what, expected, actual);
    }
}

## In short

Arrays have a size fixed when you write the code. When the size comes from the data, ask for memory while the program runs: `malloc(bytes)` returns a pointer to a new block on the **heap**, or `NULL` when there is none. The block lasts until you `free` it, even after the function that made it returns.

#### 🎞️ malloc, realloc and free

![malloc reserves a block in the heap, realloc grows it (maybe moving it), and free gives it back while the pointer keeps the old address until it is set to NULL](malloc-steps.svg)

```c
#include <stdio.h>
#include <stdlib.h>

int main(void) {
    int capacity = 4, count = 0, value;
    int *values = malloc(capacity * sizeof *values);
    if (values == NULL) return 1;
    while (scanf("%d", &value) == 1) {
        if (count == capacity) {                 // full: double it
            capacity *= 2;
            int *bigger = realloc(values, capacity * sizeof *values);
            if (bigger == NULL) { free(values); return 1; }
            values = bigger;
        }
        values[count++] = value;
    }
    printf("%d numbers\n", count);
    free(values);
    return 0;
}
```

`realloc` may move the block, so keep using the pointer it returns.

**malloc, realloc and free:** the block grows, keeps its values, and is freed at the end.

In [ ]:
int n = 4;
int *squares = malloc(n * sizeof *squares);
if (squares == NULL) {
    return 1;
}
for (int i = 0; i < n; i++) {
    squares[i] = i * i;
}
int *bigger = realloc(squares, 2 * n * sizeof *squares);
if (bigger == NULL) {
    free(squares);
    return 1;
}
squares = bigger;              /* the block may have moved */
for (int i = n; i < 2 * n; i++) {
    squares[i] = i * i;
}
for (int i = 0; i < 2 * n; i++) {
    printf("%d ", squares[i]);
}
printf("\n");
free(squares);

### ✏️ Exercise: A range in the heap

Return a new block of n ints holding 0, 1, …, n − 1. The caller frees it.

In [ ]:
int *range_array(int n) {
    // TODO: malloc n ints and fill them
    return calloc(n, sizeof(int));
}

int main(void) {
    int *r = range_array(4);
    check_int("r[0]", r[0], 0);
    check_int("r[3]", r[3], 3);
    free(r);
    return 0;
}

<details>
<summary>Show an answer</summary>

```c
int *range_array(int n) {
    int *a = malloc(n * sizeof(int));
    if (a == NULL) {
        return NULL;
    }
    for (int i = 0; i < n; i++) {
        a[i] = i;
    }
    return a;
}

int main(void) {
    int *r = range_array(4);
    check_int("r[0]", r[0], 0);
    check_int("r[3]", r[3], 3);
    free(r);
    return 0;
}
```

</details>

**Watch out:** after `free(p)`, `p` still holds the old address. Using it, or freeing it twice, is undefined behavior; memory checks catch it.

### ✏️ Exercise: Duplicate a string

Return a copy of s in a new heap block. Remember the extra byte for `'\0'`.

In [ ]:
char *duplicate(const char *s) {
    // TODO: malloc strlen(s) + 1 bytes, then strcpy
    return calloc(1, 1);
}

int main(void) {
    char *d = duplicate("hello");
    check_str("the copy", d, "hello");
    char *e = duplicate("");
    check_str("an empty copy", e, "");
    free(d);
    free(e);
    return 0;
}

<details>
<summary>Show an answer</summary>

```c
char *duplicate(const char *s) {
    char *copy = malloc(strlen(s) + 1);
    if (copy == NULL) {
        return NULL;
    }
    strcpy(copy, s);
    return copy;
}

int main(void) {
    char *d = duplicate("hello");
    check_str("the copy", d, "hello");
    char *e = duplicate("");
    check_str("an empty copy", e, "");
    free(d);
    free(e);
    return 0;
}
```

</details>

## Readings

- [Manual Memory Allocation](https://beej.us/guide/bgc/html/split/manual-memory-allocation.html) (Beej's Guide to C Programming, chapter 12, 12 min): malloc and free, error checking, calloc, and realloc, including "Reading in Lines of Arbitrary Length", which is the Longest Line challenge almost line by line.
- [Lecture 4: Copying and malloc](https://cs50.harvard.edu/x/2026/notes/4/#copying-and-malloc) (CS50x (Harvard), lecture notes, 10 min): Optional: from "Copying and malloc" to "Garbage Values": why copying a pointer doesn't copy the string, and the stack and the heap in a picture.

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **Unknown Count** · Easy
- **Longest Line** · Medium
- **Pascal's Triangle** · Medium
- **Matrix Multiply** · Hard